# VirtualDB Tutorial: Unified Cross-Dataset Queries

The `VirtualDB` class provides a unified query interface across heterogeneous datasets with different experimental condition structures and terminologies. Each dataset defines conditions in its own way, with properties at different hierarchy levels and using different naming conventions. VirtualDB uses external YAML configuration to:

- Map varying structures to a common schema
- Normalize factor level names (e.g., "D-glucose", "dextrose", "glu" all become "glucose")
- Enable cross-dataset queries with standardized field names and values

In this tutorial, we'll explore how to use VirtualDB to query and compare data across multiple datasets.

## 1. Introduction: Why VirtualDB?

### The Problem

When working with multiple genomic datasets, you'll encounter:

1. **Inconsistent terminology**: One dataset uses "D-glucose", another uses "glucose", yet another uses "glu"
2. **Different hierarchies**: Conditions stored at repo-level vs config-level vs field-level
3. **Varying structures**: Different nesting patterns for the same logical information

### The Solution

VirtualDB abstracts these differences:

- **Common schema**: Query all datasets using standardized field names
- **Automatic normalization**: "D-glucose" and "glucose" both match `carbon_source="glucose"`
- **Unified interface**: Single query retrieves data from multiple datasets

## 2. Basic Setup

VirtualDB requires a YAML configuration file that defines:
- Which datasets to include
- How to map their fields to common names
- How to normalize varying terminologies

In [1]:
from tfbpapi.virtual_db import VirtualDB
import pandas as pd

# For this tutorial, we'll create a sample configuration
# In practice, you'd load this from a YAML file
config_yaml = """
repositories:
  BrentLab/harbison_2004:
    dataset:
      harbison_2004:
        carbon_source:
          field: condition
          path: media.carbon_source.compound
        temperature_celsius:
          field: condition
          path: temperature_celsius
        environmental_condition:
          field: condition

  BrentLab/kemmeren_2014:
    dataset:
      kemmeren_2014:
        carbon_source:
          path: media.carbon_source.compound
        temperature_celsius:
          path: temperature_celsius

factor_aliases:
  carbon_source:
    glucose: [D-glucose, dextrose, glu]
    galactose: [D-galactose, gal]
    raffinose: [D-raffinose]

missing_value_labels:
  carbon_source: "unspecified"

description:
  carbon_source: The carbon source provided during growth
  temperature_celsius: Growth temperature in degrees Celsius
  environmental_condition: Named environmental condition
"""

# Save config to temporary file
import tempfile
from pathlib import Path

temp_config = Path(tempfile.mkdtemp()) / "vdb_config.yaml"
temp_config.write_text(config_yaml)

print(f"Configuration saved to: {temp_config}")

/home/chase/code/tfbp/tfbpapi/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration saved to: /tmp/tmpi90ps0mq/vdb_config.yaml


In [2]:
# Initialize VirtualDB with the configuration
vdb = VirtualDB(str(temp_config))

print("VirtualDB initialized successfully!")
print(f"Configured repositories: {len(vdb.config.repositories)}")

VirtualDB initialized successfully!
Configured repositories: 2


## 3. Schema Discovery

Before querying, let's explore what fields are available.

In [3]:
# Get all fields defined in any dataset
all_fields = vdb.get_fields()

print("All available fields:")
for field in sorted(all_fields):
    print(f"  - {field}")

All available fields:
  - carbon_source
  - environmental_condition
  - temperature_celsius


In [4]:
# Get fields present in ALL datasets (common fields)
common_fields = vdb.get_common_fields()

print("Common fields (present in all datasets):")
for field in sorted(common_fields):
    print(f"  - {field}")

Common fields (present in all datasets):
  - carbon_source
  - temperature_celsius


In [5]:
# Get fields for a specific dataset
harbison_fields = vdb.get_fields("BrentLab/harbison_2004", "harbison_2004")

print("Fields in harbison_2004:")
for field in sorted(harbison_fields):
    print(f"  - {field}")

Fields in harbison_2004:
  - carbon_source
  - environmental_condition
  - temperature_celsius


### Discovering Valid Values

VirtualDB can tell you what values exist for each field.

In [6]:
# Get all unique values for a field (normalized)
carbon_sources = vdb.get_unique_values("carbon_source")

print("Unique carbon sources (normalized):")
for source in sorted(carbon_sources):
    print(f"  - {source}")

Fetching 1 files: 100%|██████████| 1/1 [00:09<00:00,  9.60s/it]

Unique carbon sources (normalized):
  - unspecified


In [7]:
# Get values broken down by dataset
carbon_by_dataset = vdb.get_unique_values("carbon_source", by_dataset=True)

print("Carbon sources by dataset:")
for dataset, sources in carbon_by_dataset.items():
    print(f"\n{dataset}:")
    for source in sorted(sources):
        print(f"  - {source}")

Carbon sources by dataset:

BrentLab/kemmeren_2014/kemmeren_2014:
  - unspecified


## 4. Simple Queries

Now let's start querying data. The `query()` method is the primary interface.

### Basic Query: All Samples with Glucose

By default, queries return sample-level data (one row per sample) with all configured fields.

In [8]:
# Query all datasets for samples grown on glucose
glucose_samples = vdb.query(filters={"carbon_source": "glucose"})

print(f"Found {len(glucose_samples)} samples with glucose")
print(f"\nColumns: {list(glucose_samples.columns)}")
print(f"\nFirst few rows:")
glucose_samples.head()

Found 0 samples with glucose

Columns: ['sample_id', 'regulator_locus_tag', 'regulator_symbol', 'carbon_source', 'dataset_id']

First few rows:


,sample_id,regulator_locus_tag,regulator_symbol,carbon_source,dataset_id


### Query Specific Datasets

Limit your query to specific datasets using the `datasets` parameter.

In [9]:
# Query only harbison_2004
harbison_glucose = vdb.query(
    filters={"carbon_source": "glucose"},
    datasets=[("BrentLab/harbison_2004", "harbison_2004")]
)

print(f"Found {len(harbison_glucose)} samples from harbison_2004")
harbison_glucose.head()

Found 0 samples from harbison_2004


""


### Select Specific Fields

Return only the fields you need with the `fields` parameter.

In [10]:
# Get just sample_id, carbon_source, and temperature
minimal_data = vdb.query(
    filters={"carbon_source": "glucose"},
    fields=["sample_id", "carbon_source", "temperature_celsius"]
)

print(f"Columns: {list(minimal_data.columns)}")
minimal_data.head()

Columns: ['sample_id', 'carbon_source', 'dataset_id']


,sample_id,carbon_source,dataset_id


## 5. Advanced Queries

VirtualDB supports more sophisticated query patterns.

### Multiple Filter Conditions

In [11]:
# Samples with glucose at 30C
glucose_30c = vdb.query(
    filters={
        "carbon_source": "glucose",
        "temperature_celsius": 30
    }
)

print(f"Found {len(glucose_30c)} samples with glucose at 30C")
glucose_30c.head()

Found 0 samples with glucose at 30C


,sample_id,regulator_locus_tag,regulator_symbol,carbon_source,dataset_id


### Numeric Range Queries

In [12]:
# Samples at temperature >= 30C
warm_samples = vdb.query(
    filters={"temperature_celsius": (">=", 30)}
)

print(f"Found {len(warm_samples)} samples at >= 30C")

# Samples between 28C and 32C
moderate_temp = vdb.query(
    filters={"temperature_celsius": ("between", 28, 32)}
)

print(f"Found {len(moderate_temp)} samples between 28-32C")

Found 1487 samples at >= 30C
Found 1487 samples between 28-32C


### Factor Alias Expansion

When you query for a normalized value, VirtualDB automatically expands to all original aliases.

In [13]:
# Query for "galactose" matches "D-galactose", "gal", and "galactose"
galactose_samples = vdb.query(filters={"carbon_source": "galactose"})

print(f"Found {len(galactose_samples)} galactose samples")
print("\nThis query internally expanded to:")
print("  WHERE carbon_source IN ('D-galactose', 'gal', 'galactose')")

Found 0 galactose samples

This query internally expanded to:
  WHERE carbon_source IN ('D-galactose', 'gal', 'galactose')


### Complete Data Retrieval

By default, `query()` returns sample-level metadata (one row per sample). 
Set `complete=True` to get all measurements (many rows per sample).

In [14]:
# Get complete data with measurements
complete_data = vdb.query(
    filters={"carbon_source": "glucose"},
    datasets=[("BrentLab/harbison_2004", "harbison_2004")],
    complete=True
)

print(f"Complete data: {len(complete_data)} rows")
print(f"Columns: {list(complete_data.columns)}")
print("\nFirst few measurements:")
complete_data.head()

Complete data: 0 rows
Columns: []

First few measurements:


""


In [15]:
# You can combine complete=True with field selection
# Get just the binding data columns
binding_data = vdb.query(
    filters={"carbon_source": "glucose"},
    datasets=[("BrentLab/harbison_2004", "harbison_2004")],
    fields=["sample_id", "regulator_symbol", "target_symbol", "effect", "pvalue"],
    complete=True
)

print(f"Binding data: {len(binding_data)} measurements")
binding_data.head(10)

Binding data: 0 measurements


""


## 6. Understanding the Configuration

Let's examine the configuration structure in more detail.

### Property Mapping Levels

Properties can be extracted at three hierarchy levels:

1. **Repository-wide**: Common to all datasets
```yaml
repositories:
  BrentLab/some_repo:
    nitrogen_source:  # Applies to all datasets in this repo
      path: media.nitrogen_source.name
```

2. **Dataset-specific**: Specific to one config
```yaml
    dataset:
      config_name:
        phosphate_source:  # Only for this dataset
          path: media.phosphate_source.compound
```

3. **Field-level**: Varies per sample
```yaml
        carbon_source:
          field: condition  # Extract from 'condition' field
          path: media.carbon_source.compound
```

### Field Aliases (Column Renaming)

When you specify just a `field` without a `path`, it creates a column alias:

```yaml
environmental_condition:
  field: condition  # Renames 'condition' to 'environmental_condition'
```

This is useful for:
- Standardizing field names across datasets
- Enabling normalization of the field values

### Factor Aliases (Value Normalization)

The `factor_aliases` section maps varying terminologies to standard values:

```yaml
factor_aliases:
  carbon_source:
    glucose: [D-glucose, dextrose, glu]
    galactose: [D-galactose, gal]
```

This means:
- Querying for `"glucose"` matches `"D-glucose"`, `"dextrose"`, and `"glu"`
- All returned values are normalized to the canonical form (`"glucose"`)

## 7. Real-World Example: Cross-Dataset Comparison

Let's do a practical analysis comparing experiments across datasets.

In [16]:
# Compare number of samples by carbon source across datasets

# Get all samples
all_samples = vdb.query()

# Count by dataset and carbon source
summary = all_samples.groupby(['_repo_id', '_config_name', 'carbon_source']).size()
summary = summary.reset_index(name='num_samples')

print("Sample counts by dataset and carbon source:")
print(summary.to_string(index=False))

KeyError: '_repo_id'

In [ ]:
# Compare glucose experiments at different temperatures

glucose_by_temp = vdb.query(
    filters={"carbon_source": "glucose"},
    fields=["sample_id", "temperature_celsius", "environmental_condition"]
)

# Count samples by temperature
temp_counts = glucose_by_temp['temperature_celsius'].value_counts().sort_index()

print("Glucose samples by temperature:")
for temp, count in temp_counts.items():
    print(f"  {temp}C: {count} samples")

In [ ]:
# Get binding data for a specific regulator across datasets

# Query for FHL1 binding in glucose conditions
fhl1_binding = vdb.query(
    filters={
        "carbon_source": "glucose",
        "regulator_symbol": "FHL1"
    },
    fields=["sample_id", "regulator_symbol", "target_symbol", "effect", "pvalue"],
    complete=True
)

print(f"Found {len(fhl1_binding)} FHL1 binding measurements in glucose")

# Find significant targets (p < 0.001)
significant = fhl1_binding[fhl1_binding['pvalue'] < 0.001]
print(f"Significant targets: {len(significant)}")

# Top 10 by effect size
top_targets = significant.nlargest(10, 'effect')[['target_symbol', 'effect', 'pvalue']]
print("\nTop 10 targets by effect size:")
print(top_targets.to_string(index=False))

## 8. Best Practices

### 1. Use Field Selection

Only request the fields you need - it's more efficient:

```python
# Good: Only get what you need
df = vdb.query(
    filters={"carbon_source": "glucose"},
    fields=["sample_id", "temperature_celsius"]
)

# Less efficient: Get all fields
df = vdb.query(filters={"carbon_source": "glucose"})
```

### 2. Filter Before Complete Data

Use filters to reduce the dataset before getting complete data:

```python
# Good: Filter first
df = vdb.query(
    filters={"carbon_source": "glucose", "temperature_celsius": 30},
    complete=True
)

# Less efficient: Get all data then filter in pandas
df = vdb.query(complete=True)
df = df[(df['carbon_source'] == 'glucose') & (df['temperature_celsius'] == 30)]
```

### 3. Use Normalized Values

Always query using the normalized (canonical) values from your config:

```python
# Good: Use canonical value
df = vdb.query(filters={"carbon_source": "glucose"})

# Might miss data: Using original value
df = vdb.query(filters={"carbon_source": "D-glucose"})  # Only matches exact string
```

### 4. Check Available Values First

Use `get_unique_values()` to discover what values exist:

```python
# See what carbon sources are available
sources = vdb.get_unique_values("carbon_source")
print(sources)

# Then query for specific ones
df = vdb.query(filters={"carbon_source": "galactose"})
```

## Summary

VirtualDB provides a powerful abstraction for querying heterogeneous genomic datasets:

1. **Schema Discovery**: Explore available fields and values with `get_fields()` and `get_unique_values()`
2. **Unified Queries**: Query across datasets using standardized field names
3. **Automatic Normalization**: Varying terminologies automatically mapped to common values
4. **Flexible Filtering**: Support for exact matches, ranges, and complex conditions
5. **Sample or Complete Data**: Choose between metadata-only or full measurements

For more details on the configuration format and design principles, see the [Virtual Database Design](../virtual_database_concepts.md) documentation.